# Random Forest

"Trips & Travel.Com" company wants to enable and establish a viable business model to expand the customer base. One of the ways to expand the customer base is to introduce a new offering of packages. Currently, there are 5 types of packages the company is offering - Basic, Standard, Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the customers purchased the packages. However, the marketing cost was quite high because customers were contacted at random without looking at the available information. The company is now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support or increase one's sense of well-being. However, this time company wants to harness the available data of existing and potential customers to make the marketing expenditure more efficient.

In [ ]:
# importing libs:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
df = pd.read_csv("../../../../../data/Travel.csv")
df.head()

## Data Cleaning

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
non_numeric_cols = df.select_dtypes(exclude='number').columns
numeric_cols

In [ ]:
non_numeric_cols

In [ ]:
for col in non_numeric_cols:
   print(df[col].value_counts(),"\n")

In [ ]:
for col in non_numeric_cols:
   print(col ,": ", df[col].unique())

Gender and Marital Status have non uniform entries, to correct them:

In [ ]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
df['Gender'].unique()

In [ ]:
df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')
df['MaritalStatus'].unique()

In [ ]:
# missing values


In [ ]:
df.isnull().sum()[df.isnull().sum() >= 1]


In [ ]:
features_with_na = [
    features for features in df.columns if df[features].isnull().sum()>=1
]
for feature in features_with_na:
    print(feature, np.round(df[feature].isnull().mean()*100,5),"% missing values")

In [ ]:
null_percent = df.isnull().mean() * 100
null_percent = null_percent[null_percent >= 1]
null_percent.round(5)

In [ ]:
null_summary = (
    df.isnull()
      .mean()
      .mul(100)
      .round(5)
      .to_frame(name="percent_missing")
)

null_summary = null_summary[null_summary["percent_missing"] >= 1]


In [ ]:
null_summary.sort_values("percent_missing", ascending=False)


In [ ]:
df[features_with_na].select_dtypes(exclude='O').describe()

### Imputing the null values:
1. Impute Median value for Age column
2. Impute Mode for Type of Contract
3. Impute Median for Duration of Pitch
4. Impute Mode for NumberofFollowup as it is Discrete feature
5. Impute Mode for PreferredPropertyStar
6. Impute Median for NumberofTrips
7. Impute Mode for NumberOfChildrenVisiting
8. Impute Median for MonthlyIncome

In [ ]:
df.isnull().sum()[df.isnull().sum() >= 1]

In [ ]:
# df[features_with_na].unique()
for col in features_with_na:
   print(col ,": ", df[col].unique())

In [ ]:
# if discrete/ categorical we did mode else median

# deprecated way
# df['Age'].fillna(df.Age.median(), inplace=True)
# df['TypeofContact'].fillna(df.TypeofContact.mode()[0], inplace=True)
# df['DurationOfPitch'].fillna(df.DurationOfPitch.median(), inplace=True)
# df['NumberOfFollowups'].fillna(df.NumberOfFollowups.mode()[0], inplace=True)
# df['PreferredPropertyStar'].fillna(df.PreferredPropertyStar.mode()[0], inplace=True)
# df['NumberOfTrips'].fillna(df.NumberOfTrips.median(), inplace=True)
# df['NumberOfChildrenVisiting'].fillna(df.NumberOfChildrenVisiting.mode(), inplace=True)
# df['MonthlyIncome'].fillna(df.MonthlyIncome.median(), inplace=True)
# better:
# df['Age'] = df['Age'].fillna(df['Age'].median())
# df['TypeofContact'] = df['TypeofContact'].fillna(df['TypeofContact'].mode()[0])
# df['DurationOfPitch'] = df['DurationOfPitch'].fillna(df['DurationOfPitch'].median())
# df['NumberOfFollowups'] = df['NumberOfFollowups'].fillna(df['NumberOfFollowups'].mode()[0])
# df['PreferredPropertyStar'] = df['PreferredPropertyStar'].fillna(df['PreferredPropertyStar'].mode()[0])
# df['NumberOfTrips'] = df['NumberOfTrips'].fillna(df['NumberOfTrips'].median())
# df['NumberOfChildrenVisiting'] = df['NumberOfChildrenVisiting'].fillna(
#     df['NumberOfChildrenVisiting'].mode()[0]
# )
# df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
# or:
df.fillna({
    'Age': df['Age'].median(),
    'TypeofContact': df['TypeofContact'].mode()[0],
    'DurationOfPitch': df['DurationOfPitch'].median(),
    'NumberOfFollowups': df['NumberOfFollowups'].mode()[0],
    'PreferredPropertyStar': df['PreferredPropertyStar'].mode()[0],
    'NumberOfTrips': df['NumberOfTrips'].median(),
    'NumberOfChildrenVisiting': df['NumberOfChildrenVisiting'].mode()[0],
    'MonthlyIncome': df['MonthlyIncome'].median()
}, inplace=True)


In [ ]:
df.isnull().sum()

*yipeee*

In [ ]:
df.head()

In [ ]:
df.drop('CustomerID', inplace=True, axis=1)

In [ ]:
df.head()

## Feature Engineering

In [ ]:
df.head()

In [ ]:
# reducing unnecessary columns
df['Visited'] = df['NumberOfChildrenVisiting']+df["NumberOfPersonVisiting"]
df.head()

In [ ]:
df = df.drop(columns=['NumberOfChildrenVisiting', 'NumberOfPersonVisiting'])
df.head()

In [ ]:
# Numerical features:
num_features = [feature for feature in df.columns if df[feature].dtype!='O']
print(f'Number of Numerical Features: {len(num_features)}')

# categorical features:
cat_features = [feature for feature in df.columns if df[feature].dtype=='O']
print(f'Number of Categorical Features: {len(cat_features)}')

# Discrete features:
discrete_features = [
    feature for feature in num_features if len(df[feature].unique())<=25
    ]
print(f'Number of Discrete Features: {len(discrete_features)}')

# Continuous features:
con_features = [
    feature for feature in num_features if feature not in discrete_features
    ]
print(f'Number of Continuous Features: {len(con_features)}')

In [ ]:
# Numerical features:
num_features = df.select_dtypes(include='number').columns.tolist()
print(f'Number of Numerical Features: {len(num_features)}')

# categorical features:
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Number of Categorical Features: {len(cat_features)}')

# Discrete features:
discrete_features = [
    f for f in num_features if df[f].nunique() <= 25
]
print(f'Number of Discrete Features: {len(discrete_features)}')

# Continuous features:
con_features = [
    f for f in num_features if f not in discrete_features
]
print(f'Number of Continuous Features: {len(con_features)}')

In [ ]:
df.head()

## Train and Test and Model Training

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df.drop(columns="ProdTaken")
y = df['ProdTaken']

In [ ]:
X.head()

In [ ]:
y.value_counts()

In [ ]:
plt.scatter(range(len(y)), y)
plt.xlabel('Index')
plt.ylabel('y')
plt.title('Scatter plot of y')
plt.show()

In [ ]:
# Count of 0s and 1s
counts = y.value_counts()

plt.bar(counts.index, counts.values, color=['skyblue', 'salmon'])
plt.xticks([0, 1], ['No', 'Yes'])  # optional: rename 0/1 to No/Yes
plt.ylabel('Count')
plt.title('Distribution of binary target y')
plt.show()


In [ ]:
num_features

In [ ]:
for cols in num_features:
    sns.countplot(x=cols, hue=y, data=X.join(y))
    plt.ylabel('Count')
    plt.title('Distribution of y with respect to col1')
    plt.show()


In [ ]:
numeric_X = X.select_dtypes(include='number')

# Pairplot of numeric features
sns.pairplot(numeric_X)

In [ ]:
y_named = y.copy()
y_named.name = 'y'  # set the column name

sns.pairplot(numeric_X.join(y_named), hue='y')


In [ ]:
sns.heatmap(X.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")


In [ ]:
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    random_state=67,
    test_size=0.25,
)
X_train.shape , X_test.shape

## Messed up i imputed before split causing data leakage

In [ ]:
num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

In [ ]:
# setting up pipelines:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Numerical pipeline
# num_pipeline = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='median')),  # fill missing with median
#     ('scaler', StandardScaler())                    # scale features
# ])
discrete_num_features = ['Visited']
continuous_num_features = ['Age', 'DurationOfPitch', 'NumberOfTrips', 'MonthlyIncome']

# Discrete pipeline (mode)
discrete_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

# Continuous pipeline (median + scale)
continuous_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill missing with mode
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))     # convert to 0/1 vectors
])

preprocessor = ColumnTransformer([
    ('cont', continuous_pipeline, continuous_num_features),
    ('disc', discrete_pipeline, discrete_num_features),
    ('cat', cat_pipeline, cat_features)
])

In [ ]:
# models:
from sklearn.ensemble import RandomForestClassifier

models = {
    "Random Forest": RandomForestClassifier(),
}

In [ ]:
preprocessor

In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}


In [ ]:
pipelines

In [ ]:
X = df.drop('ProdTaken', axis=1)
y = df['ProdTaken']

X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25, random_state=67
    )

In [ ]:
for name, cols in [('continuous', continuous_num_features),
                   ('discrete', discrete_num_features),
                   ('categorical', cat_features)]:
    print(f"{name} columns:", cols)


In [ ]:
# now transformation:
X_train_transformed = preprocessor.fit_transform(X_train)

In [ ]:
X_train_transformed=pd.DataFrame(X_train)

In [ ]:
pd.DataFrame(X_train_transformed)

In [ ]:
X_test_transformed = pd.DataFrame(preprocessor.transform(X_test))

In [ ]:
X_test_transformed

## Training

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

def get_classification_metrics(y_true, y_pred, y_proba=None, model_name=None, verbose=True):
    """
    Calculate standard classification metrics and optionally print them.

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    y_proba : array-like, optional
        Predicted probabilities for positive class (for ROC AUC)
    model_name : str, optional
        Name of the model for printing
    verbose : bool
        Whether to print the metrics

    Returns:
    --------
    metrics : dict
        Dictionary containing accuracy, precision, recall, f1, roc_auc, confusion_matrix
    """
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred)
    metrics['recall'] = recall_score(y_true, y_pred)
    metrics['f1'] = f1_score(y_true, y_pred)
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    else:
        metrics['roc_auc'] = None

    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred)
    if verbose:
        if model_name:
            print(f"--- {model_name} ---")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"F1-score : {metrics['f1']:.4f}")
        print(f"ROC AUC  : {metrics['roc_auc']}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred))
        print("-"*40)
    return metrics


In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


- **Accuracy:** The proportion of total correct predictions (both classes). On the test set, 88.5% of samples are correctly classified.
- **Precision:** Of all predicted positives, how many were actually positive. Here, ~79.5% of predicted '1's are correct.
- **Recall (Sensitivity):** Of all actual positives, how many were correctly predicted. Only ~45.8% of positive cases were identified → many false negatives.
- **F1-score:** Harmonic mean of precision and recall; balances both. Here, 0.5808 indicates moderate performance on the positive class.
- **ROC AUC:** Probability that a randomly chosen positive ranks higher than a randomly chosen negative. 0.8923 shows good ranking ability despite low recall.
- **Confusion Matrix:** Displays counts of true negatives, false positives, false negatives, and true positives. Shows the model missed 115 positives in the test set.


In [ ]:
# models:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC()
}


In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

pipelines

In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # # TRAIN metrics
    # y_train_pred = pipe.predict(X_train)
    # try:
    #     y_train_proba = pipe.predict_proba(X_train)[:, 1]
    # except AttributeError:
    #     y_train_proba = None
    # train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:
## Hyperparameter Training
rf_params = {"max_depth": [5, 8, 15, None, 10],
"max_features": [5, 7, "sqrt", 8],
"min_samples_split": [2, 8, 15, 20],
"n_estimators": [100, 200, 500, 1000]
}

In [ ]:
# Models List for Hyperparameter tuning
randomcv_models = [
("RF", RandomForestClassifier(), rf_params),
]

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
model_param = {}
for name,model,params in randomcv_models:
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    # # Update params to target 'model' step
    params_prefixed = {f"model__{key}": value for key, value in params.items()}

    # RandomizedSearchCV
    random = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params_prefixed,
        # param_distributions=params,
        n_iter=100,
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42
    )
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

# Print best params
for model_name in model_param:
    print(f'--------------Best Param for {model_name}------------------')
    print(model_param[model_name])

In [ ]:
model_param

In [ ]:
pipelines_best = {}
for name, (step_name, model) in zip([m[0] for m in randomcv_models], [(m[1], m[1]) for m in randomcv_models]):
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    # Set the best params directly
    pipe.set_params(**model_param[name])
    pipelines_best[name] = pipe

In [ ]:
results = {}

for name, pipe in pipelines_best.items():
    print(f"Training {name} with best params...")
    # Train the model
    pipe.fit(X_train, y_train)
    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba, verbose=True)
    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba, verbose=True)
    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:
dt_params = {
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"]
}

lr_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],  # L1 requires solver='liblinear'
    "solver": ["lbfgs", "saga"],  # compatible with l2
    "class_weight": [None, "balanced"]
}

rf_params = {
    "n_estimators": [100, 200, 500, 1000],
    "max_depth": [None, 5, 8, 10, 15],
    "max_features": ["sqrt", 5, 7, 8],
    "min_samples_split": [2, 8, 15, 20],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

svm_params = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf", "poly"],
    "gamma": ["scale", "auto"],
    "class_weight": [None, "balanced"],
    "probability": [True]  # needed if you want predict_proba
}

# Combine models with their hyperparameter grids
randomcv_models = [
    ("Decision Tree", DecisionTreeClassifier(), dt_params),
    ("Logistic Regression", LogisticRegression(max_iter=1000), lr_params),
    ("Random Forest", RandomForestClassifier(random_state=42), rf_params),
    ("SVM", SVC(), svm_params)
]

In [ ]:
# model_param = {}
# for name, model,params in randomcv_models:
#     random = RandomizedSearchCV(
#         estimator=model,
#         param_distributions=params,
#         n_iter=50,   # reduce if too slow
#         cv=3,
#         verbose=2,
#         n_jobs=-1,
#         random_state=42
#     )
#     random.fit(X_train, y_train)
#     model_param[name] = random.best_params
#     for model_name in model_param:
#         print(f'--------------Best Param for {model_name}------------------')
#         print(model_param[model_name])


In [ ]:
for name,model,params in randomcv_models:
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    # # Update params to target 'model' step
    params_prefixed = {f"model__{key}": value for key, value in params.items()}

    # RandomizedSearchCV
    random = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params_prefixed,
        # param_distributions=params,
        n_iter=100,
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42,
        scoring='f1'
    )
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

# Print best params
for model_name in model_param:
    print(f'--------------Best Param for {model_name}------------------')
    print(model_param[model_name])

In [ ]:
model_param

In [ ]:
pipelines_best = {}

for name, model, params in randomcv_models:
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Set the best params from RandomizedSearchCV
    pipe.set_params(**model_param[name])

    pipelines_best[name] = pipe


In [ ]:
results = {}

for name, pipe in pipelines_best.items():
    print(f"Training {name} with best params...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba, verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba, verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:
results = {}

for name, pipe in pipelines_best.items():
    print(f"Training {name} with best params...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba, verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


Before f1 as scoring:

'''
Training Decision Tree with best params...

Accuracy : 0.8453

Precision: 0.5920

Recall   : 0.3491

F1-score : 0.4392

ROC AUC  : 0.7722422006351579

Confusion Matrix:

[[959  51]
 [138  74]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      1010
           1       0.59      0.35      0.44       212

    accuracy                           0.85      1222
   macro avg       0.73      0.65      0.67      1222
weighted avg       0.83      0.85      0.83      1222

----------------------------------------

Training Logistic Regression with best params...

Accuracy : 0.8306

Precision: 0.5581

Recall   : 0.1132

F1-score : 0.1882

ROC AUC  : 0.6893470950868672

Confusion Matrix:

[[991  19]
 [188  24]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.98      0.91      1010
           1       0.56      0.11      0.19       212

    accuracy                           0.83      1222
   macro avg       0.70      0.55      0.55      1222
weighted avg       0.79      0.83      0.78      1222

----------------------------------------

Training Random Forest with best params...

Accuracy : 0.8977

Precision: 0.8425

Recall   : 0.5047

F1-score : 0.6313

ROC AUC  : 0.9080235382028768

Confusion Matrix:
[[990  20]
 [105 107]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      1010
           1       0.84      0.50      0.63       212

    accuracy                           0.90      1222
   macro avg       0.87      0.74      0.79      1222
weighted avg       0.89      0.90      0.89      1222

----------------------------------------

Training SVM with best params...

Accuracy : 0.8625

Precision: 0.7292

Recall   : 0.3302

F1-score : 0.4545

ROC AUC  : 0.7727255744442367

Confusion Matrix:
[[984  26]
 [142  70]]

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1010
           1       0.73      0.33      0.45       212

    accuracy                           0.86      1222
   macro avg       0.80      0.65      0.69      1222
weighted avg       0.85      0.86      0.84      1222

----------------------------------------
'''



## Observation
### Decision Tree
| Metric    | Before (Accuracy) | After (F1) | Change   |
| --------- | ----------------- | ---------- | -------- |
| Accuracy  | 0.8453            | **0.8699** | ⬆        |
| Precision | 0.5920            | **0.6109** | ⬆        |
| Recall    | 0.3491            | **0.6887** | 🚀 BIG ⬆ |
| F1-score  | 0.4392            | **0.6475** | 🚀 BIG ⬆ |
| ROC AUC   | 0.7722            | **0.7983** | ⬆        |
### Logistic Regression
| Metric    | Before (Accuracy) | After (F1) | Change    |
| --------- | ----------------- | ---------- | --------- |
| Accuracy  | **0.8306**        | 0.6465     | ⬇         |
| Precision | **0.5581**        | 0.2737     | ⬇         |
| Recall    | 0.1132            | **0.6274** | 🚀 HUGE ⬆ |
| F1-score  | 0.1882            | **0.3811** | 🚀 BIG ⬆  |
| ROC AUC   | 0.6893            | 0.6943     | ≈         |

### Random Forest
| Metric    | Before (Accuracy) | After (F1) | Change |
| --------- | ----------------- | ---------- | ------ |
| Accuracy  | 0.8977            | 0.8977     | =      |
| Precision | 0.8425            | 0.8425     | =      |
| Recall    | 0.5047            | 0.5047     | =      |
| F1-score  | 0.6313            | 0.6313     | =      |
| ROC AUC   | 0.9080            | 0.9080     | =      |

### SVM
| Metric    | Before (Accuracy) | After (F1) | Change   |
| --------- | ----------------- | ---------- | -------- |
| Accuracy  | **0.8625**        | 0.7520     | ⬇        |
| Precision | **0.7292**        | 0.3767     | ⬇        |
| Recall    | 0.3302            | **0.6557** | 🚀 BIG ⬆ |
| F1-score  | 0.4545            | **0.4785** | ⬆        |
| ROC AUC   | 0.7727            | **0.7917** | ⬆        |


| Model               | Accuracy-Optimized    | F1-Optimized      | Winner               |
| ------------------- | --------------------- | ----------------- | -------------------- |
| Decision Tree       | Misses positives      | Balanced          | **F1**               |
| Logistic Regression | Nearly useless recall | Detects positives | **F1**               |
| Random Forest       | Strong                | Strong            | **Tie (RF is king)** |
| SVM                 | Conservative          | Balanced          | **F1**               |


In [ ]:
model_param

In [ ]:
pipe = pipelines_best["Random Forest"]
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score

y_train_proba_cv = cross_val_predict(
    pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]


In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
f1_scores = []

for t in thresholds:
    y_pred = (y_train_proba_cv >= t).astype(int)
    f1_scores.append(f1_score(y_train, y_pred))

best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print(f"Best threshold: {best_threshold:.2f}")
print(f"CV F1-score  : {best_f1:.4f}")

In [ ]:
y_test_proba = pipe.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(int)

get_classification_metrics(y_test, y_test_pred, y_test_proba, verbose=True)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
plt.figure(figsize=(7, 6))

auc_models = [
    {
        "label": "Random Forest",
        "pipe": pipelines_best["Random Forest"]
    },
    {
        "label": "Decision Tree",
        "pipe": pipelines_best["Decision Tree"]
    },
    {
        "label": "SVM",
        "pipe": pipelines_best["SVM"]
    }
]

for algo in auc_models:
    pipe = algo["pipe"]
    
    y_test_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    auc = roc_auc_score(y_test, y_test_proba)

    plt.plot(fpr, tpr, label=f"{algo['label']} (AUC = {auc:.3f})")

# Random baseline
plt.plot([0, 1], [0, 1], "k--")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Example for Random Forest
fpr, tpr, thresholds = roc_curve(y_train, y_train_proba_cv)

# maximize Youden's J
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)

print("ROC-based threshold:", thresholds[best_idx])


In [ ]:
for t in [0.227, 0.34, 0.5]:
    y_pred = (y_test_proba >= t).astype(int)
    print(f"\nThreshold = {t}")
    get_classification_metrics(y_test, y_pred, y_test_proba)


## Conclusion

"We optimized the decision threshold using cross-validated F1-score rather than using
the default 0.5, which significantly improved minority-class recall without excessively
increasing false positives."

In [ ]:
# ## Plot ROC AUC Curve
# from sklearn.metrics import roc_auc_score, roc_curve
# plt.figure()

# # Add the models to the list that you want to view on the ROC plot
# auc_models = [{
#     'label': 'Random Forest Classifier',
#     'model': RandomForestClassifier(n_estimators=1000,min_samples_split=2,
#     max_features=7,max_depth=None,class_weight='balanced'),

#     'auc': 0.9080235382028768
# },]
# # create loop through all model
# for algo in auc_models:
#     model = algo['model'] # select the model
#     model.fit(X_train, y_train) # train the model
#     # Compute False postive rate, and True positive rate
#     fpr, tpr, thresholds = roc_curve(y_test, model.predict_proba(X_test) [:,1
#     # Calculate Area under the curve to display on the plot
#     plt.plot(fpr, tpr, label='%s ROC (area = %0.2f)'% (algo['label'], al3
#     # Custom settings for the plot
#     plt.plot([0, 1], [0, 1],'r -- ')
#     plt.xlim([0.0, 1.0])
#     plt.ylim([0.0, 1.05])
#     plt.xlabel('1-Specificity(False Positive Rate)')
#     plt.ylabel('Sensitivity(True Positive Rate)')
#     plt.title('Receiver Operating Characteristic')
#     plt.legend(loc="lower right")
#     plt.savefig("auc.png")